# 🐍 Day 24 · Python Basics — SQL for a Shop: Searching Products (Fashion Brand Agent)

> **Goal of this notebook:** The fashion agent finds products in a MySQL database. Learn the same SQL with SQLite (no server needed): filters, sorting, counting and grouping.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. Create a products table
2. Rows as dictionaries with `row_factory`
3. `WHERE`, `LIKE`, `ORDER BY`, `LIMIT`
4. `COUNT`, `AVG` and `GROUP BY`
5. 🔗 A safe search function with optional filters

## 1. Create a products table
The real app uses **MySQL**. The SQL language is almost the same, so we practise with SQLite.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, category TEXT, color TEXT, price REAL)")
db.executemany("INSERT INTO products (name, category, color, price) VALUES (?, ?, ?, ?)", [
    ("Linen Shirt", "shirts", "white", 39.0),
    ("Oxford Shirt", "shirts", "blue", 49.0),
    ("Slim Jeans", "jeans", "blue", 59.0),
    ("Summer Dress", "dresses", "red", 79.0),
    ("Silk Dress", "dresses", "black", 149.0),
    ("Denim Jacket", "jackets", "blue", 99.0),
])
print("Products added ✅")

## 2. Rows as dictionaries
`sqlite3.Row` lets you read columns by name instead of by position.

In [ ]:
db.row_factory = sqlite3.Row
row = db.execute("SELECT * FROM products WHERE id = 1").fetchone()
print(row["name"], row["price"])
print(dict(row))

## 3. Filters, patterns and sorting
- `LIKE '%dress%'` → contains the word *dress*
- `ORDER BY price DESC` → most expensive first
- `LIMIT 2` → only 2 results

In [ ]:
def show(rows):
    for r in rows:
        print(f"  {r['name']:<14} {r['color']:<6} ${r['price']}")

print("Blue items under $60:")
show(db.execute("SELECT * FROM products WHERE color = ? AND price < ?", ("blue", 60)))

print("Name contains 'dress':")
show(db.execute("SELECT * FROM products WHERE name LIKE ?", ("%dress%",)))

print("Two most expensive:")
show(db.execute("SELECT * FROM products ORDER BY price DESC LIMIT 2"))

## 4. Counting and grouping

In [ ]:
for r in db.execute("SELECT category, COUNT(*) AS n, AVG(price) AS avg_price FROM products GROUP BY category"):
    print(f"{r['category']:<8} {r['n']} items, average ${r['avg_price']:.2f}")

## 5. 🔗 A safe search tool with optional filters
The agent may give only *some* filters. We build the query step by step, always using `?` placeholders.

In [ ]:
def search_products(category=None, color=None, max_price=None):
    sql = "SELECT * FROM products WHERE 1=1"     # 1=1 is always true, so we can add 'AND ...' easily
    params = []
    if category:
        sql += " AND category = ?"; params.append(category)
    if color:
        sql += " AND color = ?"; params.append(color)
    if max_price is not None:
        sql += " AND price <= ?"; params.append(max_price)
    return [dict(r) for r in db.execute(sql + " ORDER BY price", params)]

print(search_products(color="blue"))
print(search_products(category="dresses", max_price=100))

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Find all products that cost more than $70.
2. Count how many `blue` products there are.
3. Call `search_products(category="shirts", max_price=40)`.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
show(db.execute("SELECT * FROM products WHERE price > ?", (70,)))
print(db.execute("SELECT COUNT(*) FROM products WHERE color = ?", ("blue",)).fetchone()[0])
print(search_products(category="shirts", max_price=40))
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Rows by name | `db.row_factory = sqlite3.Row` |
| Filter | `WHERE color = ? AND price < ?` |
| Contains | `WHERE name LIKE '%dress%'` |
| Sort & limit | `ORDER BY price DESC LIMIT 2` |
| Group | `SELECT category, COUNT(*) ... GROUP BY category` |
| Optional filters | `sql += ' AND color = ?'` |

## ➡️ What's next?
You now know the Python used in **Day_24_Project_Fashion_Brand_Agent**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀